# Warfarin Precision Dosing Model

**Machine Learning Approach to Support Clinical Decision**

**Author:** Khaliq Lamid

**Project Type:** Supervised Machine Learning

**Domain:** Health Data Science / Clinical Decision Support

---


## Abstract

Warfarin is one of the most commonly prescribed blood thinners in the world but getting the dose right is genuinely difficult. Too much and the patient bleeds, too little and they clot. The correct dose varies considerably between patients depending on their age, weight, genetics and other medications which makes a one size fits all approach inadequate.

This project builds a machine learning model to predict each patient's stable Warfarin maintenance dose from their clinical and genomic profile. The pipeline merges five separate data sources on patient ID, handles missing values, engineers features like BMI and log transformed therapeutic range, encodes categorical variables and compares three regression models: Linear Regression, Random Forest and XGBoost. Performance is measured using MAE, RMSE and R².

Accuracy alone isn't enough in a clinical setting. Clinicians need to understand why a model recommends a particular dose not just that it does. So the project also uses MLflow to track experiments, LIME to generate per prediction explanations and a Gradio prototype to show how the model would sit inside an actual clinical workflow.

## Table of Contents

1. Problem Definition and Clinical Context
2. Data Loading and Initial Inspection
3. Data Wrangling: Standardisation and Merging
4. Exploratory Data Analysis
5. Feature Engineering
6. One Hot Encoding and Train/Test Split
7. Model 1: Linear Regression Baseline
8. Model 2: Random Forest Regressor
9. Model 3: XGBoost Regressor
10. Experiment Tracking with MLflow
11. Model Interpretability with LIME
12. Deployment Prototype with Gradio
13. Conclusion and Reflection


## 1. Problem Definition and Clinical Context

### Clinical Background
Warfarin prevents blood clots, but the margin between a safe dose and a harmful one is narrow. Too high and the patient bleeds; too low and clotting goes unchecked. What makes it harder is that the right dose differs substantially between patients, depending on:

- **Demographic factors**: age, sex, ethnicity
- **Clinical factors**: comorbidities, concurrent medications (e.g. amiodarone)
- **Lifestyle factors**: smoking status, alcohol intake, vitamin K diet
- **Genetic factors**: CYP2C9, VKORC1, and CYP4F2 variants which affect how quickly the body processes the drug

### Objective
Build a supervised regression model that predicts each patient's stable Warfarin dose in mg, using the factors above and pair it with explainability tools so clinicians can see the reasoning behind each prediction.

### Why this matters
Right now finding the correct dose takes weeks of trial and error. During that period, patients carry real risk. A model that predicts a reasonable starting dose from a patient's profile could shorten that window considerably. But a prediction alone isn't enough if a clinician can't follow the logic they won't act on it. Accuracy and interpretability both have to be there.


## 2. Data Loading and Initial Inspection

The dataset is provided as five separate CSV files each containing one aspect of patient information:
- patient_ids.csv — patient identifiers and demographics
- outcomes.csv — the target variable (final stable dose)
- lifestyle.csv — smoking, alcohol and diet
- clinical.csv — clinical measurements and concurrent medication
- genomics.csv — genetic markers (CYP2C9, VKORC1, CYP4F2)

These need to be merged into a single feature table before any modelling can begin.


In [1]:
import pandas as pd

### 2.1 Loading the five source datasets


In [2]:
patient_ids = pd.read_csv("patient_ids.csv")
outcomes = pd.read_csv("outcomes.csv")
lifestyle = pd.read_csv("lifestyle.csv")
clinical = pd.read_csv("clinical.csv")
genomics = pd.read_csv("genomics.csv")

### 2.2 Initial inspection of each dataset


In [3]:
patient_ids.head()
outcomes.head()
lifestyle.head()
clinical.head()
genomics.head()


,Patient_ID,CYP2C9,VKORC1,CYP4F2
0,P000001,*1/*3,A/G,C/C
1,P000002,*1/*1,A/G,C/C
2,P000003,*1/*1,A/G,C/T
3,P000004,*1/*2,G/G,C/T
4,P000005,*1/*1,G/G,C/T


## 3. Data Wrangling: Standardisation and Merging

Data from multiple sources rarely arrives clean. Before merging column names are standardised and the join key is checked across all five files to make sure nothing gets dropped or duplicated in the merge.

### 3.1 Standardising column names


Stripping whitespace, lowercasing and replacing spaces with underscores to ensure consistency across all five datasets.


In [4]:
datasets = [patient_ids, outcomes, lifestyle, clinical, genomics]

for d in datasets:
    d.columns = d.columns.str.strip().str.lower().str.replace(" ", "_")

### 3.2 Verifying the join key is present in every dataset


In [5]:
for d in datasets:
    print("patient_id" in d.columns)

True
True
True
True
True


### 3.3 Merging all five datasets on patient_id


Using patient_ids as the base table and left joining the other four datasets on patient_id. This preserves all patients in the cohort even if some have missing information in specific source files.


In [6]:
df = patient_ids.copy()

In [7]:
df = df.merge(outcomes, on="patient_id", how="left")
df.head()
df.shape

df = df.merge(lifestyle, on="patient_id", how="left")
df.head()
df.shape

df = df.merge(clinical, on="patient_id", how="left")
df.head()
df.shape

df = df.merge(genomics, on="patient_id", how="left")
df.head()
df.shape

(50000, 24)

## 4. Exploratory Data Analysis


### 4.1 Checking for missing values


In [8]:
df.isna().sum()

patient_id                           0
final_stable_dose_mg                 0
inr_stabilization_days               0
adverse_event                    41736
time_in_therapeutic_range_pct        0
alcohol_intake                   19925
smoking_status                       0
diet_vitk_intake                     0
age                                  0
sex                                  0
weight_kg                            0
height_cm                            0
ethnicity                            0
hypertension                         0
diabetes                             0
chronic_kidney_disease               0
heart_failure                        0
amiodarone                           0
antibiotics                          0
aspirin                              0
statins                              0
cyp2c9                               0
vkorc1                               0
cyp4f2                               0
dtype: int64

### 4.2 Summary statistics of numerical features


In [9]:
df.describe()

,final_stable_dose_mg,inr_stabilization_days,time_in_therapeutic_range_pct,age,weight_kg,height_cm,hypertension,diabetes,chronic_kidney_disease,heart_failure,amiodarone,antibiotics,aspirin,statins
count,50000.000000,50000.000000,50000.000000,50000.00000,50000.000000,50000.000000,50000.00000,50000.000000,50000.00000,50000.000000,50000.000000,50000.000000,50000.00000,50000.000000
mean,3.294196,7.717420,70.274858,67.43430,74.613420,169.526920,0.40302,0.200640,0.19852,0.198740,0.099740,0.099500,0.30300,0.301720
std,1.620941,2.795154,10.556340,9.88392,14.668014,9.758804,0.49051,0.400483,0.39889,0.399056,0.299656,0.299335,0.45956,0.459009
min,1.000000,0.000000,30.000000,40.00000,45.000000,150.000000,0.00000,0.000000,0.00000,0.000000,0.000000,0.000000,0.00000,0.000000
25%,2.000000,6.000000,63.100000,61.00000,64.000000,163.000000,0.00000,0.000000,0.00000,0.000000,0.000000,0.000000,0.00000,0.000000
50%,3.000000,8.000000,70.400000,67.00000,74.000000,169.000000,0.00000,0.000000,0.00000,0.000000,0.000000,0.000000,0.00000,0.000000
75%,4.400000,9.000000,77.500000,74.00000,85.000000,176.000000,1.00000,0.000000,0.00000,0.000000,0.000000,0.000000,1.00000,1.000000
max,10.100000,21.000000,95.000000,90.00000,120.000000,200.000000,1.00000,1.000000,1.00000,1.000000,1.000000,1.000000,1.00000,1.000000


### 4.3 Inspecting categorical features


Checking the distribution of key categorical variables such as sex and smoking status.


In [10]:
df.select_dtypes(include="object").columns
df['sex'].value_counts(dropna=False)
df['smoking_status'].value_counts(dropna=False)

smoking_status
Non-smoker       30108
Former Smoker    12482
Smoker            7410
Name: count, dtype: int64

## 5. Feature Engineering

Feature engineering applies clinical and statistical reasoning to transform raw variables into more informative inputs for the model.


### 5.1 Separating target and features


Separating the target final_stable_dose_mg from the predictors and dropping the patient_id since it has no predictive value.


In [11]:
y = df['final_stable_dose_mg']


In [12]:
X = df.drop(columns=['patient_id', 'final_stable_dose_mg'])


In [13]:
X.dtypes


inr_stabilization_days             int64
adverse_event                     object
time_in_therapeutic_range_pct    float64
alcohol_intake                    object
smoking_status                    object
diet_vitk_intake                  object
age                                int64
sex                               object
weight_kg                          int64
height_cm                          int64
ethnicity                         object
hypertension                       int64
diabetes                           int64
chronic_kidney_disease             int64
heart_failure                      int64
amiodarone                         int64
antibiotics                        int64
aspirin                            int64
statins                            int64
cyp2c9                            object
vkorc1                            object
cyp4f2                            object
dtype: object

### 5.2 Engineering BMI from height and weight


Body Mass Index is a recognised clinical measure that captures the relationship between height and weight in a single variable. It is more clinically meaningful than treating height and weight as separate features.

In [14]:
X['BMI'] = X['weight_kg'] / ((X['height_cm']/100) ** 2)


### 5.3 Banding age into clinical groups


Age is grouped into clinically meaningful bands (<30, 30–50, 50–70, 70+) because:
1. Clinicians typically think in age bands, not raw years
2. Warfarin sensitivity changes non linearly with age, older patients metabolise the drug differently
3. Banding can help tree based models split more cleanly


In [15]:
X['age_group'] = pd.cut(X['age'], bins=[0,30,50,70,100], labels=['<30','30-50','50-70','70+'])


### 5.4 Combining lifestyle factors


Combining smoking status and alcohol intake into a single interaction feature, since these lifestyle factors often interact in their effect on Warfarin metabolism.


In [16]:
X['smoke_alcohol'] = X['smoking_status'].astype(str) + '_' + X['alcohol_intake'].astype(str)

### 5.5 Log transforming the therapeutic range variable


The time_in_therapeutic_range_pct variable is right skewed. Applying log1p (log(1+x)) reduces skewness making the distribution more symmetric and improving the performance of linear models. Plain log breaks on zero values and log1p sidesteps that by adding 1 before taking the log, so zero inputs still produce a valid result.


In [17]:
import numpy as np

X['log_time_in_range'] = np.log1p(X['time_in_therapeutic_range_pct'])


### 5.6 One hot encoding categorical features


Converting all categorical variables to numeric using one hot encoding with drop_first=True to avoid Overlap.


In [18]:
X = pd.get_dummies(X, drop_first=True)

## 6. Train/Test Split

Splitting into 80% training and 20% testing with a fixed random state for reproducibility.


In [19]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

## 7. Model 1: Linear Regression Baseline

Linear Regression is chosen as the baseline because it is simple, interpretable and gives a clear performance floor against which more complex models can be compared.


### 7.1 Training the Linear Regression model


In [20]:
from sklearn.linear_model import LinearRegression

lr = LinearRegression()
lr.fit(X_train, y_train)

,fit_intercept,True
,copy_X,True
,tol,1e-06
,n_jobs,None
,positive,False


In [21]:
y_pred = lr.predict(X_test)

### 7.2 Evaluating Linear Regression


Evaluating using three standard regression metrics:
- **MAE (Mean Absolute Error)**: average size of prediction error in mg
- **RMSE (Root Mean Squared Error)**:penalises larger errors more heavily important in a clinical setting
- **R² (coefficient of determination)**: proportion of dose variance explained by the model


In [22]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

# Make predictions
y_pred = lr.predict(X_test)  # replace 'lr' with your trained model (e.g., rf for Random Forest)

# Calculate metrics
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

# Print results
print(f"MAE: {mae:.4f}")
print(f"RMSE: {rmse:.4f}")
print(f"R²: {r2:.4f}") 

MAE: 0.5125
RMSE: 0.6649
R²: 0.8318


## 8. Model 2: Random Forest Regressor

Random Forest is a non linear ensemble method that often handles tabular healthcare data well. it captures interactions between features without explicit engineering and is robust to outliers.


### 8.1 Re-preparing features for tree based models


Re-preparing the feature matrix and re-encoding categoricals with dummy_na=True to explicitly capture missingness as a category, which is useful for tree based models.


In [23]:
X.dtypes

inr_stabilization_days                    int64
time_in_therapeutic_range_pct           float64
age                                       int64
weight_kg                                 int64
height_cm                                 int64
hypertension                              int64
diabetes                                  int64
chronic_kidney_disease                    int64
heart_failure                             int64
amiodarone                                int64
antibiotics                               int64
aspirin                                   int64
statins                                   int64
BMI                                     float64
log_time_in_range                       float64
adverse_event_Clotting                     bool
alcohol_intake_Light                       bool
alcohol_intake_Moderate                    bool
smoking_status_Non-smoker                  bool
smoking_status_Smoker                      bool
diet_vitk_intake_Low                    

In [24]:
X_train.dtypes


inr_stabilization_days                    int64
time_in_therapeutic_range_pct           float64
age                                       int64
weight_kg                                 int64
height_cm                                 int64
hypertension                              int64
diabetes                                  int64
chronic_kidney_disease                    int64
heart_failure                             int64
amiodarone                                int64
antibiotics                               int64
aspirin                                   int64
statins                                   int64
BMI                                     float64
log_time_in_range                       float64
adverse_event_Clotting                     bool
alcohol_intake_Light                       bool
alcohol_intake_Moderate                    bool
smoking_status_Non-smoker                  bool
smoking_status_Smoker                      bool
diet_vitk_intake_Low                    

In [25]:
import pandas as pd
# Target
y = df['final_stable_dose_mg']

# Features
X = df.drop(columns=['final_stable_dose_mg', 'patient_id'])


In [26]:
categorical_cols = X.select_dtypes(include='object').columns.tolist()
print("Categorical columns:", categorical_cols)


Categorical columns: ['adverse_event', 'alcohol_intake', 'smoking_status', 'diet_vitk_intake', 'sex', 'ethnicity', 'cyp2c9', 'vkorc1', 'cyp4f2']


In [27]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


In [28]:
import pandas as pd

# Convert text columns to 0/1 numbers
X_train_encoded = pd.get_dummies(X_train, columns=categorical_cols, dummy_na=True)
X_test_encoded = pd.get_dummies(X_test, columns=categorical_cols, dummy_na=True)

# Make sure train and test have the same columns
X_test_encoded = X_test_encoded.reindex(columns=X_train_encoded.columns, fill_value=0)


### 8.2 Training the Random Forest model


In [29]:
from sklearn.ensemble import RandomForestRegressor

rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(X_train_encoded, y_train)


,n_estimators,100
,criterion,'squared_error'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,1.0
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


### 8.3 Evaluating Random Forest


In [30]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

# Make predictions
y_pred = rf.predict(X_test_encoded)

# Metrics
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print(f"MAE: {mae:.4f}")
print(f"RMSE: {rmse:.4f}")
print(f"R²: {r2:.4f}")


MAE: 0.4056
RMSE: 0.5660
R²: 0.8781


## 9. Model 3: XGBoost Regressor

XGBoost is a gradient boosted tree algorithm known for strong performance on tabular data. It builds trees sequentially where each tree corrects the errors of the previous ones and typically outperforming standard Random Forest on structured prediction problems.

### 9.1 Confirming XGBoost installation


In [33]:
import xgboost as xgb
print(xgb.__version__)  


2.1.3


### 9.2 Training and evaluating XGBoost


In [32]:
import xgboost as xgb
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

# Define the model
xgb_model = xgb.XGBRegressor(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=5,
    random_state=42
)

# Train the model
xgb_model.fit(X_train_encoded, y_train)

# Make predictions
y_pred_xgb = xgb_model.predict(X_test_encoded)

# Evaluate
mae = mean_absolute_error(y_test, y_pred_xgb)
rmse = np.sqrt(mean_squared_error(y_test, y_pred_xgb))
r2 = r2_score(y_test, y_pred_xgb)

print(f"XGBoost MAE: {mae:.4f}")
print(f"XGBoost RMSE: {rmse:.4f}")
print(f"XGBoost R²: {r2:.4f}")


XGBoost MAE: 0.3949
XGBoost RMSE: 0.5456
XGBoost R²: 0.8867


## 10. Experiment Tracking with MLflow

MLflow is used to track model parameters, evaluation metrics and artifacts across runs. This is essential for reproducibility without experiment tracking, comparing dozens of model variants becomes unreliable and informal.

### 10.1 Importing MLflow


In [34]:
import mlflow
import mlflow.sklearn
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

### 10.2 Preparing features for the tracked XGBoost run


In [35]:
import pandas as pd

categorical_cols = ['adverse_event','alcohol_intake','smoking_status',
                    'diet_vitk_intake','sex','ethnicity','cyp2c9','vkorc1','cyp4f2']

# One-hot encode
X_encoded = pd.get_dummies(X, columns=categorical_cols, dummy_na=True)

# Split train/test
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X_encoded, y, test_size=0.2, random_state=42)

### 10.3 Logging the XGBoost run with parameters and metrics


In [36]:
with mlflow.start_run(run_name="XGBoost_Model"):

    # Log model parameters
    mlflow.log_param("model_type", "XGBoost")
    mlflow.log_param("n_estimators", xgb_model.n_estimators)
    mlflow.log_param("max_depth", xgb_model.max_depth)
    mlflow.log_param("learning_rate", xgb_model.learning_rate)

    # Train the model
    xgb_model.fit(X_train, y_train)

    # Make predictions
    y_pred = xgb_model.predict(X_test)

    # Calculate metrics
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)

    # Print metrics
    print(f"XGBoost Results:\nMAE: {mae:.4f}\nRMSE: {rmse:.4f}\nR²: {r2:.4f}")

    # Log metrics in MLflow
    mlflow.log_metric("MAE", mae)
    mlflow.log_metric("RMSE", rmse)
    mlflow.log_metric("R2", r2)

    # Log the trained model
    mlflow.sklearn.log_model(xgb_model, artifact_path="model")

2025/12/16 15:01:45 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2025/12/16 15:01:45 INFO mlflow.store.db.utils: Updating database tables
2025/12/16 15:01:45 INFO alembic.runtime.migration: Context impl SQLiteImpl.
2025/12/16 15:01:45 INFO alembic.runtime.migration: Will assume non-transactional DDL.
2025/12/16 15:01:45 INFO alembic.runtime.migration: Context impl SQLiteImpl.
2025/12/16 15:01:45 INFO alembic.runtime.migration: Will assume non-transactional DDL.
2025/12/16 15:01:45 WARNING mlflow.utils.git_utils: Failed to import Git (the Git executable is probably not on your PATH), so Git SHA is not available. Error: Failed to initialize: Bad git executable.
The git executable must be specified in one of the following ways:
    - be included in your $PATH
    - be set via $GIT_PYTHON_GIT_EXECUTABLE
    - explicitly set via git.refresh(<full-path-to-git-executable>)

All git commands will error until this is rectified.

This initial message can be silenced or 

XGBoost Results:
MAE: 0.3949
RMSE: 0.5456
R²: 0.8867


## 11. Model Interpretability with LIME

LIME explains individual predictions by looking at how the model behaves around a single patient's data. Rather than explaining the model as a whole, it answers a narrower question: for this specific patient, which features drove this specific prediction

That distinction matters in a clinical setting. A clinician won't act on a dose recommendation they can't follow. LIME gives them a reason, not just a number.


### 11.1 Building the LIME explainer


In [37]:
import lime
import lime.lime_tabular

# Create a LIME explainer for regression
explainer = lime.lime_tabular.LimeTabularExplainer(
    training_data=np.array(X_train_encoded),  # your encoded training features
    feature_names=X_train_encoded.columns.tolist(),
    mode='regression'
)

### 11.2 Generating an explanation for one patient


Explaining a single prediction by showing the top features that pushed the predicted dose up or down for that specific patient, and exporting the explanation as an HTML file for clinical review.


In [38]:
# Pick an instance to explain (e.g., first test row)
i = 0
exp = explainer.explain_instance(
    data_row=X_test_encoded.iloc[i].values,  # single patient features
    predict_fn=xgb_model.predict,           # model's prediction function
    num_features=10                          # number of top features to show
)

# Save explanation as HTML using UTF-8 encoding
html = exp.as_html(show_table=True)
with open("lime_explanation.html", "w", encoding="utf-8") as f:
    f.write(html)

print(" LIME explanation saved as lime_explanation.html with UTF-8 encoding. Open this file in a browser to view it.")


✅ LIME explanation saved as lime_explanation.html with UTF-8 encoding. Open this file in a browser to view it.


## 12. Deployment Prototype with Gradio

A lightweight Gradio interface simulates what clinical use would actually look like. A clinician enters the patient's details across demographic, lifestyle, clinical and genetic fields and gets a predicted dose back. It's a proof of concept, not a finished product but it shows the model can sit inside a usable workflow rather than just a notebook.

### 12.1 Defining the prediction function


In [40]:
import pandas as pd

def predict_dose(age, sex, ethnicity, cyp2c9, vkorc1, amiodarone, smoking_status, alcohol_intake, diet_vitk_intake):
    # Creating a single row DataFrame like your training data
    input_df = pd.DataFrame({
        "age": [age],
        "sex": [sex],
        "ethnicity": [ethnicity],
        "cyp2c9": [cyp2c9],
        "vkorc1": [vkorc1],
        "amiodarone": [amiodarone],
        "smoking_status": [smoking_status],
        "alcohol_intake": [alcohol_intake],
        "diet_vitk_intake": [diet_vitk_intake]
    })

    # Encoding categorical variables 
    input_encoded = encoder.transform(input_df)  # if you used a OneHotEncoder or similar

    # Making prediction
    pred = xgb_model.predict(input_encoded)[0]
    
    return round(pred, 2)

### 12.2 Launching the Gradio interface


In [44]:
import gradio as gr
import pandas as pd
import numpy as np

# Example: your trained XGBoost model
# xgb_model

# Feature columns used in training
feature_columns = X_train.columns.tolist()

# Function to preprocess input and predict
def predict_dose(sex, age, ethnicity, cyp2c9, vkorc1, smoking_status, alcohol_intake, diet_vitk_intake, amiodarone):
    input_df = pd.DataFrame([{
        "sex": sex,
        "age": float(age),
        "ethnicity": ethnicity,
        "cyp2c9": cyp2c9,
        "vkorc1": vkorc1,
        "smoking_status": smoking_status,
        "alcohol_intake": alcohol_intake,
        "diet_vitk_intake": diet_vitk_intake,
        "amiodarone": amiodarone
    }])
    
    # One-hot encode categorical variables like in training
    input_encoded = pd.get_dummies(input_df)
    
    # Add missing columns with 0s
    for col in feature_columns:
        if col not in input_encoded.columns:
            input_encoded[col] = 0
            
    # Reorder columns to match training
    input_encoded = input_encoded[feature_columns]
    
    # Predict dose
    dose = xgb_model.predict(input_encoded)[0]
    
    return round(dose, 2)

# Gradio interface with dropdowns
inputs = [
    gr.Dropdown(choices=["M", "F"], label="Sex"),
    gr.Number(label="Age"),
    gr.Dropdown(choices=["African American", "Asian", "Caucasian", "Other"], label="Ethnicity"),
    gr.Dropdown(choices=["*1/*1", "*1/*2", "*1/*3", "*2/*2", "*2/*3", "*3/*3"], label="CYP2C9"),
    gr.Dropdown(choices=["G/G", "A/G", "A/A"], label="VKORC1"),
    gr.Dropdown(choices=["Never", "Former", "Current"], label="Smoking Status"),
    gr.Dropdown(choices=["None", "Low", "Moderate", "High"], label="Alcohol Intake"),
    gr.Dropdown(choices=["Low", "Moderate", "High"], label="Diet Vitamin K Intake"),
    gr.Checkbox(label="Amiodarone")
]

outputs = gr.Textbox(label="Predicted Dose")

gr.Interface(fn=predict_dose, inputs=inputs, outputs=outputs, title=" Warfarin Dose Predictor").launch()

* Running on local URL:  http://127.0.0.1:7863
* To create a public link, set `share=True` in `launch()`.


## 13. Conclusion and Reflection

### Summary
This project built an end to end pipeline to predict personalised Warfarin doses from patient data spanning five separate sources. The work covered merging and cleaning those sources, engineering clinically grounded features, training and comparing three regression models, tracking experiments with MLflow, explaining individual predictions with LIME and prototyping a usable interface in Gradio.

### Key Outcomes
- Built a working clinical decision support prototype that takes structured patient information and returns a personalised predicted dose.
- Compared three regression approaches and demonstrated the value of gradient boosting on this kind of structured clinical data.
- Integrated LIME for individual prediction explainability recognising that clinical adoption requires transparency not just accuracy.
- Used MLflow for experiment tracking making the work reproducible and the model versions comparable.

### Limitations
- The dataset is synthetic rather than from a real clinical trial, so none of the results carry clinical validation.
- Random Forest and XGBoost both ran on default hyperparameters tuning either could shift the performance comparison.
- Interpretability here is LIME only, SHAP values would give a different and complementary view of feature importance.
- The Gradio prototype is a demonstration because actual clinical deployment needs regulatory approval, Electronic health integration and extensive validation at a scale this project doesn't attempt.

### Reflection
The most important lesson from this project was that in healthcare data science, building an accurate model is only part of the work making the model trustable to clinicians is what actually determines whether the work gets used in practice. Prioritising interpretability through LIME and packaging the model behind a clear interface via Gradio, reflects that priority.
